Cell 1 - Google Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Cell 2 - Import Libraries

In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt

Cell 3 - Dataset Path

In [ ]:
dataset_path = "/content/drive/MyDrive/Colab Notebooks/chest_xray"

train_path = os.path.join(dataset_path, "train")
test_path = os.path.join(dataset_path, "test")
val_path = os.path.join(dataset_path, "val")

Cell 4 - Verify Dataset Structure

In [ ]:
for split in ["train", "test", "val"]:

    print(f"\n{split.upper()}")

    split_path = os.path.join(dataset_path, split)

    for cls in os.listdir(split_path):

        class_path = os.path.join(split_path, cls)

        if os.path.isdir(class_path):

            total = len([
                f for f in os.listdir(class_path)
                if f.lower().endswith((".jpeg", ".jpg", ".png"))
            ])

            print(f"{cls:<12}: {total}")



TRAIN
PNEUMONIA   : 3875
NORMAL      : 1341

TEST
NORMAL      : 234
PNEUMONIA   : 390

VAL
NORMAL      : 8
PNEUMONIA   : 8


Cell 5 - Create Metadata

In [ ]:
records = []

for split in ["train", "test", "val"]:

    split_path = os.path.join(dataset_path, split)

    for cls in os.listdir(split_path):

        class_path = os.path.join(split_path, cls)

        if not os.path.isdir(class_path):
            continue

        for file in os.listdir(class_path):

            if not file.lower().endswith((".jpeg", ".jpg", ".png")):
                continue

            records.append({
                "Split": split,
                "Class": cls,
                "Filename": file,
                "File_Path": os.path.join(class_path, file)
            })

df = pd.DataFrame(records)

Cell 6 - Dataset Statistics

In [ ]:
print("Total Images:", len(df))

print("\nClass Distribution")

print(df["Class"].value_counts())

print("\nOriginal Split Distribution")

print(df["Split"].value_counts())

Total Images: 5856

Class Distribution
Class
PNEUMONIA    4273
NORMAL       1583
Name: count, dtype: int64

Original Split Distribution
Split
train    5216
test      624
val        16
Name: count, dtype: int64


Cell 7 - Check Filename Pattern

In [ ]:
print("NORMAL Samples")

display(df[df["Class"]=="NORMAL"].head(10))

print("\nPNEUMONIA Samples")

display(df[df["Class"]=="PNEUMONIA"].head(10))

NORMAL Samples


,Split,Class,Filename,File_Path
3875,train,NORMAL,IM-0537-0001.jpeg,/content/drive/MyDrive/Colab Notebooks/chest_x...
3876,train,NORMAL,IM-0533-0001.jpeg,/content/drive/MyDrive/Colab Notebooks/chest_x...
3877,train,NORMAL,IM-0523-0001.jpeg,/content/drive/MyDrive/Colab Notebooks/chest_x...
3878,train,NORMAL,IM-0528-0001.jpeg,/content/drive/MyDrive/Colab Notebooks/chest_x...
3879,train,NORMAL,IM-0523-0001-0003.jpeg,/content/drive/MyDrive/Colab Notebooks/chest_x...
3880,train,NORMAL,IM-0522-0001.jpeg,/content/drive/MyDrive/Colab Notebooks/chest_x...
3881,train,NORMAL,IM-0519-0001.jpeg,/content/drive/MyDrive/Colab Notebooks/chest_x...
3882,train,NORMAL,IM-0531-0001-0001.jpeg,/content/drive/MyDrive/Colab Notebooks/chest_x...
3883,train,NORMAL,IM-0533-0001-0002.jpeg,/content/drive/MyDrive/Colab Notebooks/chest_x...
3884,train,NORMAL,IM-0532-0001.jpeg,/content/drive/MyDrive/Colab Notebooks/chest_x...



PNEUMONIA Samples


,Split,Class,Filename,File_Path
0,train,PNEUMONIA,person564_bacteria_2344.jpeg,/content/drive/MyDrive/Colab Notebooks/chest_x...
1,train,PNEUMONIA,person567_bacteria_2353.jpeg,/content/drive/MyDrive/Colab Notebooks/chest_x...
2,train,PNEUMONIA,person543_bacteria_2284.jpeg,/content/drive/MyDrive/Colab Notebooks/chest_x...
3,train,PNEUMONIA,person551_bacteria_2311.jpeg,/content/drive/MyDrive/Colab Notebooks/chest_x...
4,train,PNEUMONIA,person545_bacteria_2287.jpeg,/content/drive/MyDrive/Colab Notebooks/chest_x...
5,train,PNEUMONIA,person552_bacteria_2315.jpeg,/content/drive/MyDrive/Colab Notebooks/chest_x...
6,train,PNEUMONIA,person566_virus_1106.jpeg,/content/drive/MyDrive/Colab Notebooks/chest_x...
7,train,PNEUMONIA,person55_bacteria_260.jpeg,/content/drive/MyDrive/Colab Notebooks/chest_x...
8,train,PNEUMONIA,person560_bacteria_2330.jpeg,/content/drive/MyDrive/Colab Notebooks/chest_x...
9,train,PNEUMONIA,person53_bacteria_254.jpeg,/content/drive/MyDrive/Colab Notebooks/chest_x...


Cell 8 - Save Metadata

In [ ]:
df.to_csv("dataset_metadata.csv", index=False)

print("Metadata Saved Successfully")

Metadata Saved Successfully


Day 2 -Cell 1: Import **Libraries**

In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split

Cell 2: Create 3-Class Target

In [ ]:
def get_target(filename, original_class):

    if original_class == "NORMAL":
        return "NORMAL"

    filename = filename.lower()

    if "_bacteria_" in filename:
        return "BACTERIA"

    elif "_virus_" in filename:
        return "VIRUS"

    else:
        return "UNKNOWN"


df["Target"] = df.apply(
    lambda row: get_target(row["Filename"], row["Class"]),
    axis=1
)

print(df["Target"].value_counts())

Target
BACTERIA    2780
NORMAL      1583
VIRUS       1493
Name: count, dtype: int64


Cell 3: Split Dataset (70%)

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["Target"],
    random_state=42
)

print("Train Images :", len(train_df))
print("Temp Images  :", len(temp_df))

Train Images : 4099
Temp Images  : 1757


Cell 4: Split Validation & Test (15% / 15%)

In [ ]:
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["Target"],
    random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train Images      :", len(train_df))
print("Validation Images :", len(val_df))
print("Test Images       :", len(test_df))

Train Images      : 4099
Validation Images : 878
Test Images       : 879


Cell 5: Check Class Distribution

In [ ]:
print("\nTRAIN")
print(train_df["Target"].value_counts())

print("\nVALIDATION")
print(val_df["Target"].value_counts())

print("\nTEST")
print(test_df["Target"].value_counts())


TRAIN
Target
BACTERIA    1946
NORMAL      1108
VIRUS       1045
Name: count, dtype: int64

VALIDATION
Target
BACTERIA    417
NORMAL      237
VIRUS       224
Name: count, dtype: int64

TEST
Target
BACTERIA    417
NORMAL      238
VIRUS       224
Name: count, dtype: int64


Cell 6: Create Output Folder

In [ ]:
output_path = "/content/drive/MyDrive/Colab Notebooks/processed_dataset"

classes = ["NORMAL", "BACTERIA", "VIRUS"]
splits = ["train", "val", "test"]

for split in splits:
    for cls in classes:
        os.makedirs(
            os.path.join(output_path, split, cls),
            exist_ok=True
        )

print("Folder structure created successfully!")

Folder structure created successfully!


Cell 7: Copy Train Images

In [ ]:
for _, row in train_df.iterrows():

    destination = os.path.join(
        output_path,
        "train",
        row["Target"],
        row["Filename"]
    )

    shutil.copy2(row["File_Path"], destination)

print("Train images copied successfully!")

Train images copied successfully!


Cell 8: Copy Validation Images

In [ ]:
for _, row in val_df.iterrows():

    destination = os.path.join(
        output_path,
        "val",
        row["Target"],
        row["Filename"]
    )

    shutil.copy2(row["File_Path"], destination)

print("Validation images copied successfully!")

Validation images copied successfully!


Cell 9: Copy Test Images

In [ ]:
for _, row in test_df.iterrows():

    destination = os.path.join(
        output_path,
        "test",
        row["Target"],
        row["Filename"]
    )

    shutil.copy2(row["File_Path"], destination)

print("Test images copied successfully!")

Test images copied successfully!


Cell 10: Save Metadata

In [ ]:
train_df.to_csv("train_metadata.csv", index=False)
val_df.to_csv("validation_metadata.csv", index=False)
test_df.to_csv("test_metadata.csv", index=False)

print("Metadata saved successfully!")

Metadata saved successfully!


Cell 11: Final Verification

In [ ]:
for split in ["train", "val", "test"]:

    print(f"\n{split.upper()}")

    split_path = os.path.join(output_path, split)

    total = 0

    for cls in classes:

        count = len(os.listdir(os.path.join(split_path, cls)))

        print(f"{cls:<10}: {count}")

        total += count

    print("Total:", total)


TRAIN
NORMAL    : 1108
BACTERIA  : 1946
VIRUS     : 1045
Total: 4099

VAL
NORMAL    : 237
BACTERIA  : 417
VIRUS     : 224
Total: 878

TEST
NORMAL    : 238
BACTERIA  : 417
VIRUS     : 224
Total: 879


Cell 7: Verify Class Distribution

In [ ]:
print("\nTRAIN")
print(train_df["Target"].value_counts())

print("\nVALIDATION")
print(val_df["Target"].value_counts())

print("\nTEST")
print(test_df["Target"].value_counts())


TRAIN
Target
BACTERIA    1946
NORMAL      1108
VIRUS       1045
Name: count, dtype: int64

VALIDATION
Target
BACTERIA    417
NORMAL      237
VIRUS       224
Name: count, dtype: int64

TEST
Target
BACTERIA    417
NORMAL      238
VIRUS       224
Name: count, dtype: int64


Cell 8: Create Folder Structure

In [ ]:
output_path = "/content/drive/MyDrive/Colab Notebooks/processed_dataset"

classes = ["NORMAL", "BACTERIA", "VIRUS"]
splits = ["train", "val", "test"]

for split in splits:
    for cls in classes:
        os.makedirs(
            os.path.join(output_path, split, cls),
            exist_ok=True
        )

print("Folder structure created successfully!")

Folder structure created successfully!


Cell 8: Copy Validation Images

Cell 9: Copy Train Images

In [ ]:
for _, row in train_df.iterrows():

    destination = os.path.join(
        output_path,
        "train",
        row["Target"],
        row["Filename"]
    )

    shutil.copy2(row["File_Path"], destination)

print("Train images copied successfully!")

Train images copied successfully!


In [ ]:
print(df.columns)

Index(['Split', 'Class', 'Filename', 'File_Path', 'Target'], dtype='object')


In [ ]:
print(df.head())

   Split      Class                      Filename  \
0  train  PNEUMONIA  person564_bacteria_2344.jpeg   
1  train  PNEUMONIA  person567_bacteria_2353.jpeg   
2  train  PNEUMONIA  person543_bacteria_2284.jpeg   
3  train  PNEUMONIA  person551_bacteria_2311.jpeg   
4  train  PNEUMONIA  person545_bacteria_2287.jpeg   

                                           File_Path    Target  
0  /content/drive/MyDrive/Colab Notebooks/chest_x...  BACTERIA  
1  /content/drive/MyDrive/Colab Notebooks/chest_x...  BACTERIA  
2  /content/drive/MyDrive/Colab Notebooks/chest_x...  BACTERIA  
3  /content/drive/MyDrive/Colab Notebooks/chest_x...  BACTERIA  
4  /content/drive/MyDrive/Colab Notebooks/chest_x...  BACTERIA  


In [ ]:
print("merged_df shape:", merged_df.shape)

merged_df shape: (5856, 5)


In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["Target"],
    random_state=42
)

print("Train:", len(train_df))
print("Temp:", len(temp_df))

Train: 4099
Temp: 1757


In [ ]:
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["Target"],
    random_state=42
)

print("Validation:", len(val_df))
print("Test:", len(test_df))

Validation: 878
Test: 879


In [ ]:
import os

base_path = "/content/drive/MyDrive/Colab Notebooks/processed_dataset"

for split in ["train", "val", "test"]:

    print(f"\n{split.upper()}")

    total = 0

    for cls in ["NORMAL", "BACTERIA", "VIRUS"]:

        folder = os.path.join(base_path, split, cls)

        count = len(os.listdir(folder))

        print(f"{cls:<10}: {count}")

        total += count

    print("Total:", total)


TRAIN
NORMAL    : 1108
BACTERIA  : 1946
VIRUS     : 1045
Total: 4099

VAL
NORMAL    : 237
BACTERIA  : 417
VIRUS     : 224
Total: 878

TEST
NORMAL    : 238
BACTERIA  : 417
VIRUS     : 224
Total: 879
